# 🚀 Lab 28: Rapid Dashboards with Streamlit

### 📘 Lab Overview
In this lab, you will learn how to build rapid interactive dashboards using **Streamlit**. You will work with a realistic **supply chain analytics dataset**, import and manipulate it with **Pandas**, create interactive filters using **selectboxes** and **sliders**, build dynamic charts using **Matplotlib**, and launch a working dashboard that updates in real time based on user input.

This lab has been adapted for **Google Colab**, so all files, sample datasets, scripts, and dashboard apps are created directly inside the notebook.

## 🎯 Learning Objectives
By the end of this lab, students will be able to:
* Understand the fundamentals of Streamlit for creating interactive web applications
* Import and manipulate supply chain data using Pandas
* Create interactive widgets including selectboxes and sliders
* Build dynamic charts and visualizations using Matplotlib
* Launch a functional dashboard that responds to user input in real time
* Apply data visualization best practices for supply chain analytics
* Create optimized and simulated real-time dashboard versions

## 🧰 Prerequisites
Before starting this lab, students should have:
* Basic knowledge of Python programming
* Familiarity with Pandas for data manipulation
* Understanding of Matplotlib for creating charts
* Basic understanding of notebook execution in Google Colab
* No prior experience with Streamlit required

## ⚙️ Environment Setup
### 💡 ELI10: Setting up our digital workshop
Before we build our dashboard, we need to bring in our tools. Think of this like installing the right apps on your phone so you can start working. We are installing Streamlit (the dashboard builder) and LocalTunnel (the bridge that lets us see our app on the internet).

Since Streamlit apps normally run as local web servers, in Colab we will:
1. Install required packages
2. Create the dataset and dashboard scripts using notebook cells
3. Launch Streamlit in the background
4. Expose the app using **LocalTunnel**

In [ ]:
# Install the core libraries needed for data analysis and web app creation
# -q means 'quiet' to keep the output clean
%pip install -q streamlit pandas matplotlib numpy

# Install LocalTunnel globally using npm (Node Package Manager)
# This allows us to expose the Streamlit server running inside Colab to a public URL
!npm install -g localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 59.8 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋
added 22 packages in 3s
⠋
⠋3 packages are looking for funding
⠋  run `npm fund` for details
⠋

# Task 1: Environment Preparation and Data Import

## Subtask 1.1: Verify Installation and Create Working Environment
We need to make sure everything installed correctly before we move forward.

In [ ]:
import sys
# Check the Python version to ensure compatibility
print("Python version:", sys.version)

try:
    import streamlit
    import pandas
    import matplotlib
    import numpy
    print("All required packages are available and ready to use!")
except ImportError as e:
    print(f"Error: {e}. Some packages are missing.")

Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
All required packages are available and ready to use!


## 📦 Creating Supply Chain Data
### 💡 ELI10: Making 'Fake' Real Data
We need some data to practice with! We are writing a script that creates a year's worth of imaginary supply chain records (like orders, delivery times, and quality scores) so we have a 'real-world' scenario to analyze.

In [ ]:
%%writefile create_sample_data.py
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Set random seed for reproducible results so everyone gets the same 'random' data
np.random.seed(42)

def create_supply_chain_data():
    """Generate realistic supply chain analytics data."""
    start_date = datetime.now() - timedelta(days=365)
    dates = pd.date_range(start=start_date, periods=365, freq='D')

    categories = ['Electronics', 'Clothing', 'Food', 'Books', 'Home & Garden']
    suppliers = ['Supplier_A', 'Supplier_B', 'Supplier_C', 'Supplier_D', 'Supplier_E']

    data = []

    for date in dates:
        for category in categories:
            # Simulate 2 suppliers per category per day
            chosen_suppliers = np.random.choice(suppliers, size=2, replace=False)
            for supplier in chosen_suppliers:
                units_ordered = np.random.randint(50, 500)
                units_delivered = np.random.randint(40, min(480, units_ordered + 50))

                record = {
                    'Date': date,
                    'Category': category,
                    'Supplier': supplier,
                    'Units_Ordered': units_ordered,
                    'Units_Delivered': units_delivered,
                    'Cost_Per_Unit': round(np.random.uniform(5, 100), 2),
                    'Delivery_Days': np.random.randint(1, 14),
                    'Quality_Score': round(np.random.uniform(7, 10), 2)
                }
                data.append(record)

    df = pd.DataFrame(data)

    # Derived metrics to calculate business performance
    df['Total_Cost'] = df['Units_Ordered'] * df['Cost_Per_Unit']
    df['Delivery_Performance'] = df['Units_Delivered'] / df['Units_Ordered']
    df['Month'] = pd.to_datetime(df['Date']).dt.strftime('%Y-%m')

    return df

if __name__ == "__main__":
    # Run the function and save the result to a CSV file
    supply_chain_df = create_supply_chain_data()
    supply_chain_df.to_csv('supply_chain_data.csv', index=False)
    print(f"Created supply chain dataset with {len(supply_chain_df)} records")

Writing create_sample_data.py


In [ ]:
# Execute the script we just wrote to generate the CSV file
!python create_sample_data.py

Created supply chain dataset with 3650 records


## Subtask 1.3: Verify Data Import
We'll run a quick check to make sure the file we created is readable and has the columns we expect.

In [ ]:
%%writefile test_import.py
import pandas as pd

# Load the data to ensure the CSV is valid
df = pd.read_csv('supply_chain_data.csv')

print("Dataset Shape:", df.shape)
print("\nColumn Names:", df.columns.tolist())
print("\nFirst 5 rows:")
print(df.head())

Writing test_import.py


In [ ]:
!python test_import.py

Dataset Shape: (3650, 11)

Column Names: ['Date', 'Category', 'Supplier', 'Units_Ordered', 'Units_Delivered', 'Cost_Per_Unit', 'Delivery_Days', 'Quality_Score', 'Total_Cost', 'Delivery_Performance', 'Month']

First 5 rows:
                         Date     Category  ... Delivery_Performance    Month
0  2025-04-16 16:01:16.797912  Electronics  ...             0.495868  2025-04
1  2025-04-16 16:01:16.797912  Electronics  ...             1.138686  2025-04
2  2025-04-16 16:01:16.797912     Clothing  ...             0.531034  2025-04
3  2025-04-16 16:01:16.797912     Clothing  ...             1.239437  2025-04
4  2025-04-16 16:01:16.797912         Food  ...             0.432323  2025-04

[5 rows x 11 columns]


## 🛠 Building the Dashboard
### 💡 ELI10: Creating the App logic
Now we write the actual code for the dashboard. This code tells Streamlit how to show the title, how to load the data, and how to create the buttons (widgets) the user will click to change what they see.

In [ ]:
%%writefile dashboard.py
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime

# Configure the page appearance
st.set_page_config(
    page_title="Supply Chain Dashboard",
    page_icon="📦",
    layout="wide"
)

st.title("📦 Supply Chain Analytics Dashboard")
st.markdown("---")

@st.cache_data
def load_data():
    """Load and prepare the supply chain data efficiently using caching."""
    df = pd.read_csv("supply_chain_data.csv")
    df["Date"] = pd.to_datetime(df["Date"])
    if "Month" not in df.columns:
        df["Month"] = df["Date"].dt.strftime("%Y-%m")
    return df

try:
    df = load_data()
    st.success(f"✅ Data loaded successfully! {len(df)} records found.")
except FileNotFoundError:
    st.error("❌ supply_chain_data.csv not found.")
    st.stop()

# --- KPI Metrics Section ---
st.subheader("📊 Dataset Overview")
col1, col2, col3, col4 = st.columns(4)
with col1: st.metric("Total Records", len(df))
with col2: st.metric("Categories", df["Category"].nunique())
with col3: st.metric("Suppliers", df["Supplier"].nunique())
with col4: st.metric("Date Range", f"{df['Date'].min().date()} to {df['Date'].max().date()}")

# --- Filters Section ---
st.markdown("---")
st.subheader("🎛 Interactive Filters")
filter_col1, filter_col2, filter_col3 = st.columns(3)

with filter_col1:
    selected_category = st.selectbox("Select Product Category:", ["All"] + sorted(df["Category"].unique().tolist()))
with filter_col2:
    selected_supplier = st.selectbox("Select Supplier:", ["All"] + sorted(df["Supplier"].unique().tolist()))
with filter_col3:
    selected_month = st.selectbox("Select Month:", ["All"] + sorted(df["Month"].unique().tolist(), reverse=True))

# --- Slider Filters ---
slider_col1, slider_col2 = st.columns(2)
with slider_col1:
    quality_range = st.slider("Quality Score Range:", float(df["Quality_Score"].min()), float(df["Quality_Score"].max()), (float(df["Quality_Score"].min()), float(df["Quality_Score"].max())))
with slider_col2:
    delivery_range = st.slider("Delivery Days Range:", int(df["Delivery_Days"].min()), int(df["Delivery_Days"].max()), (int(df["Delivery_Days"].min()), int(df["Delivery_Days"].max())))

# Apply filters to dataframe
filtered_df = df.copy()
if selected_category != "All": filtered_df = filtered_df[filtered_df["Category"] == selected_category]
if selected_supplier != "All": filtered_df = filtered_df[filtered_df["Supplier"] == selected_supplier]
if selected_month != "All": filtered_df = filtered_df[filtered_df["Month"] == selected_month]
filtered_df = filtered_df[(filtered_df["Quality_Score"].between(quality_range[0], quality_range[1])) & (filtered_df["Delivery_Days"].between(delivery_range[0], delivery_range[1]))]

st.info(f"Final filtered dataset: {len(filtered_df)} records")

# --- Visualization Section ---
st.subheader("📈 Interactive Charts")
if not filtered_df.empty:
    chart_type = st.radio("Select Chart Type:", ["Category Analysis", "Supplier Performance", "Time Series"], horizontal=True)
    fig, ax = plt.subplots(figsize=(10, 4))

    if chart_type == "Category Analysis":
        filtered_df.groupby("Category")["Units_Ordered"].sum().plot(kind='bar', ax=ax, color='skyblue')
        ax.set_title("Total Units Ordered by Category")
    elif chart_type == "Supplier Performance":
        ax.scatter(filtered_df["Delivery_Performance"], filtered_df["Quality_Score"], alpha=0.5)
        ax.set_xlabel("Delivery Performance"); ax.set_ylabel("Quality Score")
    elif chart_type == "Time Series":
        filtered_df.groupby("Month")["Total_Cost"].sum().plot(kind='line', marker='o', ax=ax)
        ax.set_title("Total Cost Over Time")

    st.pyplot(fig)
else:
    st.warning("No data available for the selected filters.")

Writing dashboard.py


## 📡 Launching the Dashboard
To see your app, run the following two cells.
1. The first runs Streamlit in the background.
2. The second creates a public link.

**Note:** When you click the link, you might see a 'Tunnel' landing page. Simply click 'Continue' or 'Visit Site' to see your dashboard.

In [ ]:
# Start Streamlit on port 8501 and save logs to a file
!streamlit run dashboard.py --server.port 8501 >/content/streamlit_basic.log 2>&1 &

In [ ]:
# Expose the local port 8501 to the web via LocalTunnel
!lt --port 8501

your url is: https://silver-sloths-vanish.loca.lt
^C


## ⚡ Optimized Dashboard
### 💡 ELI10: Making it Faster and Cleaner
In this version, we move the filters to a 'Sidebar' on the left so the main screen stays clean for charts. We also use special Python tricks (caching) to make the data loading lightning fast.

In [ ]:
%%writefile optimized_dashboard.py
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt

st.set_page_config(page_title="Optimized Supply Chain", layout="wide")

@st.cache_data
def load_data():
    df = pd.read_csv("supply_chain_data.csv")
    df["Date"] = pd.to_datetime(df["Date"])
    return df

df = load_data()

st.sidebar.header("🎛 Dashboard Filters")
selected_cat = st.sidebar.multiselect("Categories:", df["Category"].unique(), default=df["Category"].unique())

filtered_df = df[df["Category"].isin(selected_cat)]

st.title("⚡ Optimized Supply Chain Dashboard")
col1, col2 = st.columns(2)

with col1:
    st.metric("Active Records", len(filtered_df))
    fig, ax = plt.subplots()
    filtered_df.groupby("Category")["Quality_Score"].mean().plot(kind='barh', ax=ax)
    st.pyplot(fig)

with col2:
    st.dataframe(filtered_df.head(10), use_container_width=True)

Writing optimized_dashboard.py


In [ ]:
# Launch Optimized Version on port 8502
!streamlit run optimized_dashboard.py --server.port 8502 >/content/streamlit_opt.log 2>&1 &
!lt --port 8502

## 📡 Real-time Dashboard
### 💡 ELI10: Live Data Simulation
Real dashboards often track live deliveries. Since we don't have a live factory, we'll write a script that 'invents' new data every few seconds to show how the charts move in real-time.

In [ ]:
%%writefile realtime_dashboard.py
import streamlit as st
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import time
import matplotlib.pyplot as plt

st.title("📡 Real-time Supply Chain Simulation")

if 'data' not in st.session_state:
    st.session_state.data = pd.read_csv("supply_chain_data.csv").tail(50)

# Button to simulate a new day of data
if st.button("Simulate New Day"):
    new_row = st.session_state.data.iloc[-1:].copy()
    new_row['Date'] = pd.to_datetime(new_row['Date']) + timedelta(days=1)
    new_row['Units_Ordered'] = np.random.randint(100, 600)
    st.session_state.data = pd.concat([st.session_state.data, new_row], ignore_index=True)

st.metric("Total Orders (Live)", int(st.session_state.data['Units_Ordered'].sum()))
st.line_chart(st.session_state.data.set_index('Date')['Units_Ordered'])

Writing realtime_dashboard.py


In [ ]:
# Launch Real-time Version on port 8503
!streamlit run realtime_dashboard.py --server.port 8503 >/content/streamlit_rt.log 2>&1 &
!lt --port 8503

your url is: https://hot-cities-jam.loca.lt
^C


# ✅ Verification Checklist
* [ ] Libraries installed successfully.
* [ ] `supply_chain_data.csv` created and verified.
* [ ] `dashboard.py` runs and is accessible via LocalTunnel.
* [ ] Filters (selectbox/slider) correctly update the metrics and charts.
* [ ] Optimized and Real-time versions are functional.

# 🛠 Troubleshooting
* **Tunnel not loading?** Check the logs in the text file created in `/content/` or restart the `!lt` cell.
* **Command not found?** Ensure the `%pip install` cell ran without errors.
* **Port Busy?** Use `!pkill -f streamlit` to reset all background apps.

# 🧠 Key Concepts Summary
* **Streamlit**: Turns Python scripts into web apps instantly.
* **Widgets**: The interactive components like buttons and sliders.
* **State Management**: Keeping track of user input (like our Real-time data simulation).
* **LocalTunnel**: A tool to share apps from inside a closed environment like Colab.

# 🏁 Conclusion
You have successfully built a suite of Supply Chain Dashboards! This skill is vital in the real world for tracking inventory, monitoring supplier quality, and presenting business insights to stakeholders in an interactive way.